# 🗼 Watchtower Server — Google Colab Deploy

This notebook starts the Watchtower headless server inside a Colab session and exposes it publicly via **ngrok**.

**Steps:**
1. Set your `API_KEY` and `NGROK_TOKEN` in the form below
2. Run all cells (`Runtime → Run all`)
3. Copy the ngrok URL printed at the end
4. Paste it into the Watchtower app → Settings → Remote Server

> ⚠️ The server runs only while this Colab tab is open. For persistent hosting use Railway or Render.

In [ ]:
# @title ⚙️ Configuration { display-mode: "form" }
API_KEY = "changeme"  # @param {type:"string"}
NGROK_TOKEN = ""       # @param {type:"string"} Get a free token at https://dashboard.ngrok.com/
PORT = 8080            # @param {type:"integer"}

In [ ]:
# @title 1️⃣ Install Node.js 20
import subprocess, sys

def run(cmd, **kwargs):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

print("Installing Node.js 20...")
run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash -")
run("apt-get install -y nodejs")
print("Node version:", run("node --version").strip())
print("npm  version:", run("npm --version").strip())

In [ ]:
# @title 2️⃣ Clone the repo & install dependencies
import os

SERVER_DIR = "/content/watchtower-server"

if not os.path.exists(SERVER_DIR):
    run("git clone --depth=1 https://github.com/ferelking242/watchtower-real.git /content/watchtower-real")
    run(f"cp -r /content/watchtower-real/app/watchtower/server {SERVER_DIR}")
else:
    print("Server dir already exists, skipping clone.")

print("Installing npm packages...")
run(f"npm ci --omit=dev", cwd=SERVER_DIR)
print("Done!")

In [ ]:
# @title 3️⃣ Start the Watchtower server
import subprocess, os, time

env = {
    **os.environ,
    "API_KEY": API_KEY,
    "PORT": str(PORT),
    "CACHE_DIR": "/content/wt-cache",
    "PREFS_DIR": "/content/wt-prefs",
}
os.makedirs("/content/wt-cache", exist_ok=True)
os.makedirs("/content/wt-prefs", exist_ok=True)

proc = subprocess.Popen(
    ["node", "server.js"],
    cwd=SERVER_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

time.sleep(3)
if proc.poll() is not None:
    out = proc.stdout.read().decode()
    raise RuntimeError(f"Server crashed on startup:\n{out}")

# Quick health check
import urllib.request
resp = urllib.request.urlopen(f"http://localhost:{PORT}/api/ping")
print("Health check:", resp.read().decode())
print(f"\n✅ Server running on port {PORT}")

In [ ]:
# @title 4️⃣ Expose via ngrok
run("pip install pyngrok -q")
from pyngrok import ngrok, conf

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
else:
    print("⚠️  No ngrok token set. Get a free one at https://dashboard.ngrok.com/")
    print("   Without a token, tunnels are limited to 1 connection / 40 req/min.")

tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("\n" + "="*60)
print(f"🗼 Watchtower Server URL:  {public_url}")
print(f"🔑 API Key:               {API_KEY}")
print("="*60)
print("\nPaste the URL above into the Watchtower app:")
print("  Settings → Remote Server → Server URL")
print("\nKeep this Colab tab open — the server stops when it closes.")

In [ ]:
# @title 5️⃣ Keep alive (optional — run this to prevent Colab from timing out)
import time
print("Server is running. Press the stop button (■) to shut down.")
try:
    while True:
        time.sleep(60)
        # Print a dot every minute so Colab knows we're alive
        print(".", end="", flush=True)
except KeyboardInterrupt:
    print("\nShutting down...")
    proc.terminate()
    ngrok.kill()
    print("Done.")